# 30-minute greenhouse forecast at 5-minute cadence

This notebook trains a two-output feed-forward model from the timestamped, processed CSV produced by `preprocess_greenhouse.py`. For the selected commercial tomato dataset, use only the 2018 and 2019 `CENTRAL.xlsx` measurements, which are recorded at 5-minute intervals. The source timestamps have no verified timezone: preserve them as naive timestamps and keep `timestamp_timezone` set to `unknown`; do not convert them to UTC or a guessed local zone.

Each input is four measured temperature/relative-humidity points at t-15, t-10, t-5, and t. The target is the measured temperature and relative humidity at exactly t+30 minutes. No intermediate values are interpolated or missing timestamps filled. The ESP32 may continue one-minute monitoring, but matching model inference is planned every five minutes.

Run the preprocessor on the two central-station workbooks and save the output as `data/processed_greenhouse.csv`. The train/validation/test ranges are chronological and all scaling is fitted on training data only. The final cell evaluates float and full-INT8 candidates and writes artifacts under `model_training/artifacts/`; it does not overwrite firmware headers. This notebook does not create synthetic data or train a Botrytis-risk classifier.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
SAMPLE_INTERVAL_MINUTES = 5
WINDOW_MINUTES = 15
WINDOW_SIZE = WINDOW_MINUTES // SAMPLE_INTERVAL_MINUTES + 1
HORIZON_MINUTES = 30
TIMESTAMP_COLUMN = "timestamp"
TARGET_TIMESTAMP_COLUMN = "forecast_target_timestamp"
FEATURE_COLUMNS = ["temperature_c", "relative_humidity_pct"]
TARGET_COLUMNS = ["target_temperature_30m_c", "target_relative_humidity_30m_pct"]
TARGET_LABELS = ["Temperature (C)", "Relative humidity (%)"]

repo_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (path for path in repo_candidates
     if (path / "model_training" / "preprocess_greenhouse.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the OmniNode_AI repository.")

PROCESSED_PATH = REPO_ROOT / "data" / "processed_greenhouse.csv"
ARTIFACT_DIR = REPO_ROOT / "model_training" / "artifacts"
if not PROCESSED_PATH.is_file():
    raise FileNotFoundError(
        "No processed dataset found at {}. Run preprocess_greenhouse.py on the "
        "2018 and 2019 CENTRAL.xlsx files with --timestamp-timezone unknown "
        "and --sample-interval-minutes 5 first.".format(PROCESSED_PATH)
    )

frame = pd.read_csv(PROCESSED_PATH)
required_columns = [
    TIMESTAMP_COLUMN,
    "timestamp_timezone",
    "sample_interval_minutes",
    "forecast_window_15min_complete",
    *FEATURE_COLUMNS,
    TARGET_TIMESTAMP_COLUMN,
    *TARGET_COLUMNS,
]
missing_columns = [name for name in required_columns if name not in frame.columns]
if missing_columns:
    raise ValueError("Processed CSV is missing columns: {}".format(missing_columns))

if not frame["timestamp_timezone"].astype(str).str.lower().eq("unknown").all():
    raise ValueError("This dataset's timezone must remain marked unknown.")
frame["sample_interval_minutes"] = pd.to_numeric(
    frame["sample_interval_minutes"], errors="raise"
)
if not frame["sample_interval_minutes"].eq(SAMPLE_INTERVAL_MINUTES).all():
    raise ValueError("Processed data must retain the native 5-minute cadence.")
frame[TIMESTAMP_COLUMN] = pd.to_datetime(frame[TIMESTAMP_COLUMN], errors="raise")
frame[TARGET_TIMESTAMP_COLUMN] = pd.to_datetime(
    frame[TARGET_TIMESTAMP_COLUMN], errors="raise"
)
if frame[TIMESTAMP_COLUMN].dt.tz is not None or frame[TARGET_TIMESTAMP_COLUMN].dt.tz is not None:
    raise ValueError("Timezone-aware timestamps are not allowed for this dataset.")
frame = frame.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)
if frame[TIMESTAMP_COLUMN].duplicated().any():
    raise ValueError("Processed CSV contains duplicate timestamps.")
expected_targets = frame[TIMESTAMP_COLUMN] + pd.Timedelta(minutes=HORIZON_MINUTES)
if not frame[TARGET_TIMESTAMP_COLUMN].eq(expected_targets).all():
    raise ValueError("Processed target timestamps must be exactly +30 minutes.")
window_flag_values = frame["forecast_window_15min_complete"].astype(str).str.lower()
if not window_flag_values.isin(["true", "false"]).all():
    raise ValueError("15-minute completeness flags must be true or false.")
print("Loaded {} timezone-unknown readings from {}".format(len(frame), PROCESSED_PATH))


In [ ]:
sample_interval_ns = pd.Timedelta(minutes=SAMPLE_INTERVAL_MINUTES).value
timestamps_ns = frame[TIMESTAMP_COLUMN].array.asi8
if not np.all(np.diff(timestamps_ns) > 0):
    raise ValueError("Timestamps must be strictly increasing.")

features = frame[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
targets = frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)
window_flags = (
    frame["forecast_window_15min_complete"].astype(str).str.lower().eq("true").to_numpy()
)

def build_partition(start, stop):
    partition_lookup = {
        timestamp: index
        for index, timestamp in enumerate(
            frame[TIMESTAMP_COLUMN].iloc[start:stop], start=start
        )
    }
    windows = []
    labels = []
    for anchor in range(start + WINDOW_SIZE - 1, stop):
        first = anchor - WINDOW_SIZE + 1
        if not np.all(
            np.diff(timestamps_ns[first : anchor + 1]) == sample_interval_ns
        ):
            continue
        if not window_flags[anchor]:
            continue

        target_time = (
            frame.at[anchor, TIMESTAMP_COLUMN]
            + pd.Timedelta(minutes=HORIZON_MINUTES)
        )
        target_index = partition_lookup.get(target_time)
        if target_index is None:
            continue
        target = targets[anchor]
        if not np.isfinite(target).all():
            continue
        if not np.allclose(target, features[target_index], rtol=0.0, atol=1e-5):
            raise ValueError("A +30-minute target does not match its timestamp.")
        window = features[first : anchor + 1]
        if not np.isfinite(window).all():
            continue
        windows.append(window)
        labels.append(target)

    return np.asarray(windows, dtype=np.float32), np.asarray(labels, dtype=np.float32)

row_count = len(frame)
train_stop = int(row_count * 0.70)
validation_stop = int(row_count * 0.85)
if not (0 < train_stop < validation_stop < row_count):
    raise ValueError("Not enough chronological rows for 70/15/15 data splits.")

X_train_raw, y_train_raw = build_partition(0, train_stop)
X_validation_raw, y_validation_raw = build_partition(train_stop, validation_stop)
X_test_raw, y_test_raw = build_partition(validation_stop, row_count)
if min(len(X_train_raw), len(X_validation_raw), len(X_test_raw)) == 0:
    raise ValueError(
        "At least one chronological split has no valid 15-minute window with an "
        "exact +30-minute target. No model was trained."
    )

x_mean = X_train_raw.reshape(-1, len(FEATURE_COLUMNS)).mean(axis=0)
x_scale = X_train_raw.reshape(-1, len(FEATURE_COLUMNS)).std(axis=0)
x_scale = np.where(x_scale == 0.0, 1.0, x_scale)
y_mean = y_train_raw.mean(axis=0)
y_scale = y_train_raw.std(axis=0)
y_scale = np.where(y_scale == 0.0, 1.0, y_scale)

X_train_scaled = ((X_train_raw - x_mean) / x_scale).astype(np.float32)
X_validation_scaled = ((X_validation_raw - x_mean) / x_scale).astype(np.float32)
X_test_scaled = ((X_test_raw - x_mean) / x_scale).astype(np.float32)
y_train_scaled = ((y_train_raw - y_mean) / y_scale).astype(np.float32)
y_validation_scaled = ((y_validation_raw - y_mean) / y_scale).astype(np.float32)
y_test_scaled = ((y_test_raw - y_mean) / y_scale).astype(np.float32)

print("Valid windows - train: {}, validation: {}, test: {}".format(
    len(X_train_raw), len(X_validation_raw), len(X_test_raw)
))
print("Input and target scaling was fitted on training windows only.")


In [ ]:
tf.keras.utils.set_random_seed(SEED)
model = keras.Sequential(
    [
        keras.Input(shape=(WINDOW_SIZE, len(FEATURE_COLUMNS))),
        layers.Flatten(),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(len(TARGET_COLUMNS)),
    ]
)
model.compile(optimizer="adam", loss="mse")

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_validation_scaled, y_validation_scaled),
    epochs=100,
    batch_size=32,
    shuffle=False,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        )
    ],
    verbose=1,
)


In [ ]:
def regression_metrics(y_true, y_pred):
    results = {}
    for index, label in enumerate(TARGET_LABELS):
        error = y_pred[:, index] - y_true[:, index]
        results[label] = {
            "MAE": float(np.mean(np.abs(error))),
            "RMSE": float(np.sqrt(np.mean(np.square(error)))),
        }
    return results


def print_metrics(title, metrics):
    print(title)
    for label, values in metrics.items():
        print("{}: MAE={:.4f}, RMSE={:.4f}".format(
            label, values["MAE"], values["RMSE"]
        ))

float_prediction_scaled = model.predict(X_test_scaled, verbose=0)
y_test_prediction_float = float_prediction_scaled * y_scale + y_mean
float_metrics = regression_metrics(y_test_raw, y_test_prediction_float)
print_metrics("Float model test metrics (physical units)", float_metrics)


In [ ]:
import json

representative_indices = np.linspace(
    0, len(X_train_scaled) - 1,
    num=min(128, len(X_train_scaled)),
    dtype=int,
)

def representative_dataset():
    for index in representative_indices:
        yield [X_train_scaled[index : index + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
int8_model = converter.convert()

interpreter = tf.lite.Interpreter(model_content=int8_model)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
if input_detail["dtype"] != np.int8 or output_detail["dtype"] != np.int8:
    raise RuntimeError("Converter did not produce fully INT8 input and output tensors.")
if tuple(input_detail["shape"]) != (1, WINDOW_SIZE, len(FEATURE_COLUMNS)):
    raise RuntimeError("Unexpected INT8 model input shape: {}".format(input_detail["shape"]))
if tuple(output_detail["shape"]) != (1, len(TARGET_COLUMNS)):
    raise RuntimeError("Unexpected INT8 model output shape: {}".format(output_detail["shape"]))

input_scale, input_zero_point = input_detail["quantization"]
output_scale, output_zero_point = output_detail["quantization"]
if input_scale <= 0.0 or output_scale <= 0.0:
    raise RuntimeError("INT8 input/output quantization scales must be positive.")

def quantize_tensor(values, detail):
    scale, zero_point = detail["quantization"]
    limits = np.iinfo(detail["dtype"])
    quantized = np.rint(values / scale + zero_point)
    return np.clip(quantized, limits.min, limits.max).astype(detail["dtype"])

int8_prediction_scaled = []
for sample in X_test_scaled:
    interpreter.set_tensor(
        input_detail["index"], quantize_tensor(sample[np.newaxis, ...], input_detail)
    )
    interpreter.invoke()
    output = interpreter.get_tensor(output_detail["index"]).astype(np.float32)
    int8_prediction_scaled.append((output - output_zero_point) * output_scale)

y_test_prediction_int8 = np.asarray(int8_prediction_scaled).reshape(-1, len(TARGET_COLUMNS))
y_test_prediction_int8 = y_test_prediction_int8 * y_scale + y_mean
int8_metrics = regression_metrics(y_test_raw, y_test_prediction_int8)
print_metrics("Full-INT8 model test metrics (physical units)", int8_metrics)
for label in TARGET_LABELS:
    print("{} INT8 delta: MAE={:+.4f}, RMSE={:+.4f}".format(
        label,
        int8_metrics[label]["MAE"] - float_metrics[label]["MAE"],
        int8_metrics[label]["RMSE"] - float_metrics[label]["RMSE"],
    ))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "omninode_forecaster_int8.tflite"
header_path = ARTIFACT_DIR / "omninode_forecaster_int8.h"
metadata_path = ARTIFACT_DIR / "forecast_preprocessing.json"
model_bytes = bytes(int8_model)
model_path.write_bytes(model_bytes)
byte_lines = [
    "  " + ", ".join("0x{:02x}".format(value) for value in model_bytes[start : start + 12])
    for start in range(0, len(model_bytes), 12)
]
header = "\n".join(
    [
        "#pragma once",
        "#include <cstddef>",
        "",
        "alignas(16) const unsigned char omninode_forecaster_int8[] = {",
        ",\n".join(byte_lines),
        "};",
        "constexpr std::size_t omninode_forecaster_int8_len = sizeof(omninode_forecaster_int8);",
        "",
    ]
)
header_path.write_text(header, encoding="utf-8")
metadata = {
    "input_window_samples": WINDOW_SIZE,
    "input_window_duration_minutes": WINDOW_MINUTES,
    "sampling_interval_minutes": SAMPLE_INTERVAL_MINUTES,
    "forecast_horizon_minutes": HORIZON_MINUTES,
    "timestamp_timezone": "unknown",
    "input_feature_order": FEATURE_COLUMNS,
    "input_mean": [float(value) for value in x_mean],
    "input_scale": [float(value) for value in x_scale],
    "output_order": TARGET_COLUMNS,
    "output_mean": [float(value) for value in y_mean],
    "output_scale": [float(value) for value in y_scale],
    "input_quantization": {"scale": float(input_scale), "zero_point": int(input_zero_point)},
    "output_quantization": {"scale": float(output_scale), "zero_point": int(output_zero_point)},
    "source_data": str(PROCESSED_PATH.relative_to(REPO_ROOT)),
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Full-INT8 model bytes: {}".format(len(model_bytes)))
print("Candidate artifacts: {}, {}, {}".format(model_path, header_path, metadata_path))
